In [ ]:
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('./data/retailmax.csv')

In [ ]:
features = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']]

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

In [ ]:
type(scaled_features)

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300, n_init=10, random_state=42)
clusters = kmeans.fit_predict(scaled_features)

In [ ]:
clusters

In [ ]:
len(clusters)

In [ ]:
kmeans = KMeans(n_clusters=5, init='k-means++', max_iter=300, n_init=10, random_state=42)
clusters = kmeans.fit_predict(scaled_features)

In [ ]:
clusters

In [ ]:
kmeans = KMeans(n_clusters=4, init='k-means++', max_iter=300, n_init=10, random_state=42)
clusters = kmeans.fit_predict(scaled_features)

In [ ]:
clusters

In [ ]:
df

In [ ]:
df['clasificación'] = clusters
df

In [ ]:
df['clasificación'].value_counts()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='clasificación', palette='Set1')
plt.title('Clusters de clientes')
plt.show()

In [ ]:
# Función para calcular la Suma de Cuadrados Dentro del Cluster (WCSS)
def calcular_wcss(datos):
    wcss = []
    for n in range(1, 11):
        kmeans = KMeans(n_clusters=n, init='k-means++', max_iter=300, n_init=10, random_state=42)
        kmeans.fit(datos)
        wcss.append(kmeans.inertia_)
    return wcss

# Calcular el WCSS para diferentes números de clusters
wcss = calcular_wcss(scaled_features)

# Graficar el método del codo
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', color='pink', linewidth=2, markersize=8, 
         markerfacecolor='pink', markeredgecolor='black', markeredgewidth=1.5)
plt.fill_between(range(1, 11), wcss, color='pink', alpha=0.3)
plt.title('Método del codo')
plt.xlabel('Número de clusters')
plt.ylabel('WCSS')
plt.grid(alpha=0.3, linestyle='--')
plt.show()

In [ ]:
# Aplicar KMeans con 5 clusters (optimo segun el metodo del codo)
kmeans = KMeans(n_clusters=5, init='k-means++', max_iter=300, n_init=10, random_state=42)
clusters = kmeans.fit_predict(scaled_features)

# Agregar al DataFrame
df['clasificación'] = clusters

# Grafica final con 5 clusters
plt.figure(figsize=(10, 6))
palette = ['#FF69B4', '#FF1493', '#FFB6C1', '#DB7093', '#C71585']
from matplotlib.colors import ListedColormap

scatter = plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], 
                      c=df['clasificación'], cmap=ListedColormap(palette),
                      s=100, alpha=0.7, edgecolors='black', linewidth=0.8)

# Centroides
centroides_scaled = kmeans.cluster_centers_
centroides = scaler.inverse_transform(centroides_scaled)
plt.scatter(centroides[:, 1], centroides[:, 2], 
            c='black', s=300, marker='X', edgecolors='white', linewidth=2, label='Centroides')

plt.title('Clusters de Clientes - Ingreso Anual vs Puntuacion de Gasto (K=5)', fontsize=14, fontweight='bold')
plt.xlabel('Ingreso Anual (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(title='Cluster', loc='best')
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

# Mostrar estadisticas por cluster
print('='*70)
print('ESTADISTICAS POR CLUSTER (K=5)')
print('='*70)
print(df.groupby('clasificación')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean())
print('\nDistribucion:')
print(df['clasificación'].value_counts().sort_index())